# The memory cap's verify-on-return: what it costs

Fork #9 (parawanderer/cpython) runs the memory cap's hook check (`mem_verify`, three probe allocations) again whenever
an approved call into C returns, so a leaf that drops the hook mid-run cannot leave the cell uncounted. #9 chose the
FULL probe over a variant that probes only when some domain's top allocator changed (`lab/mem-recheck-gate`), on native
numbers. Under wasm32 an indirect call is a larger share of a tight loop, so the trade may read differently; this
notebook keeps the rounds so it can be revisited. Ruled 2026-10-21 (repl-specs): the full probe stands, since the gate
is a proxy for "ours is in the chain"; a narrower split (the full probe only after a call that is not one of CPython's
own builtins) is Shane's to take or not.

Shapes (`shapes.py`): `trivial` (a `return 1` run: the entry, which the fix should not move), `unlifted_c` (200k `len()`
calls), `lifted_leaf` (200k calls of a lifted host leaf), `py_func` (200k calls of the cell's own def: not checked, the
control for the call path). A shape's figure for one process is the MIN over its reps; across rounds, the median of those.

In [1]:
import hashlib, json
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "python-dialect" / "verify-cost"
DATA = HERE / "data"
SHAPES = ["trivial", "unlifted_c", "lifted_leaf", "py_func"]
pd.set_option("display.width", 160)
pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16]}
              for p in sorted(DATA.iterdir())])

,file,bytes,sha256
0,cells-rounds-release-loaded.json,2428296,50a034d8697f7e09
1,cells-rounds-release.json,2416068,b50ba7f35a52d778
2,cells-rounds-trial.json,2427784,11590c0f5e8abbdd
3,native-full.jsonl,7680,70eb561c5633ad68
4,native-gate.jsonl,4260,eb66768855d183f2
5,wasm-rounds-release-loaded.json,773561,c84d6445dcda4824
6,wasm-rounds-release.json,755759,4077f49e2c59a707
7,wasm-rounds-trial.json,773197,a2f6b01a3366ed1d


## Native (macOS arm64), as in #9's body

8 interleaved rounds of 7c40b038 (`base`) against #9 (`fixed`), then 6 rounds against the changed-top variant (`gate`),
a fresh process per build per round. The ratio is taken per round, then summarised. Taken under the same stray load
as the `trial` set below: the ratios are interleaved, the absolute ns soft.

In [2]:
def native(path):
    return pd.DataFrame([json.loads(l) for l in open(path)])

def summarise(df, base, other, value):
    rows = []
    for s in [x for x in SHAPES if x in set(df["shape"])]:   # df.shape is the frame's dimensions
        b = df[(df["shape"] == s) & (df.build == base)].set_index("round")[value]
        o = df[(df["shape"] == s) & (df.build == other)].set_index("round")[value]
        r = (o / b).dropna()
        rows.append({"shape": s, f"{base} ns": b.median(), f"{other} ns": o.median(), "ratio median": r.median(),
                     "ratio min": r.min(), "ratio max": r.max(), "rounds": len(r)})
    return pd.DataFrame(rows).round(3)

display(summarise(native(DATA / "native-full.jsonl"), "base", "fixed", "ns_per_call_min"))
summarise(native(DATA / "native-gate.jsonl"), "base", "gate", "ns_per_call_min")

,shape,base ns,fixed ns,ratio median,ratio min,ratio max,rounds
0,trivial,73145.95,73687.95,1.008,0.965,1.020,8
1,unlifted_c,202.55,259.45,1.261,1.226,1.354,8
2,lifted_leaf,251.35,306.90,1.204,1.021,1.308,8
3,py_func,168.25,166.60,0.992,0.875,1.135,8


,shape,base ns,gate ns,ratio median,ratio min,ratio max,rounds
0,unlifted_c,219.15,245.20,1.126,1.081,1.227,6
1,lifted_leaf,268.20,301.55,1.137,1.040,1.198,6
2,py_func,183.70,187.55,1.019,0.965,1.122,6


## wasm32 (Pyodide under Node), every rep

`drive.mjs`: one fresh Pyodide per core per round, the order rotated each round. `base` and `base_aa` load the SAME
7c40b038 core: their ratio is the noise floor. `full` is #9 (in the `trial` set a core built from its branch commit
8575f359, in the `release` set its released core); `gate` is the variant built from `lab/mem-recheck-gate` with the
same recipe. Each set is summarised on its own: rounds are paired only within a set.

The `trial` and `release-loaded` sets ran with six stray CPU-load shells on the laptop (another session's repro, running
from 2026-10-20 until 15:22 on 2026-10-21; the janitor's mail `20261021-1330Z-flaky-ci-to-repl-specs-orphan-load-shells-killed.md`).
Every core of a set ran under the same load, so their ratios stand, but their A/A spread is wider than the laptop's own
and their absolute ns are soft. `release` was taken after the shells were killed, at a load average of about 3.

In [3]:
sets = {}
for n in ("release", "release-loaded", "trial"):
    p = DATA / f"wasm-rounds-{n}.json"
    if p.exists():
        sets[n] = pd.DataFrame([{"round": r["round"], "build": r["label"], "core": r["core"], "shape": s, "ns": min(v),
                                 "reps": len(v)} for r in json.loads(p.read_text()) for s, v in r["ns"].items()])
tables = {}
for n, wasm in sets.items():
    print(f"=== {n}:", wasm.groupby("build").core.first().to_dict(), "rounds:", wasm["round"].nunique())
    for o in ["base_aa", "full", "gate"]:
        if o in set(wasm.build):
            tables[n, o] = summarise(wasm, "base", o, "ns")
            print(f"{n}: {o} / base")
            display(tables[n, o])

=== release: {'base': 'core-7c40b038', 'base_aa': 'core-7c40b038', 'full': 'core-0169c4fd', 'gate': 'core-9c7e5fb8'} rounds: 8
release: base_aa / base


,shape,base ns,base_aa ns,ratio median,ratio min,ratio max,rounds
0,trivial,57937.500,57667.000,0.999,0.972,1.051,8
1,unlifted_c,208.744,208.343,0.994,0.975,1.019,8
2,lifted_leaf,241.254,237.383,0.988,0.953,1.033,8
3,py_func,156.845,157.459,1.006,0.969,1.050,8


release: full / base


,shape,base ns,full ns,ratio median,ratio min,ratio max,rounds
0,trivial,57937.500,60979.500,1.047,1.032,1.117,8
1,unlifted_c,208.744,283.174,1.358,1.326,1.405,8
2,lifted_leaf,241.254,313.594,1.303,1.260,1.373,8
3,py_func,156.845,164.497,1.048,1.019,1.139,8


release: gate / base


...

,shape,base ns,base_aa ns,ratio median,ratio min,ratio max,rounds
0,trivial,153250.000,143458.000,1.002,0.866,1.027,8
1,unlifted_c,496.448,492.692,0.962,0.918,1.098,8
2,lifted_leaf,571.491,574.449,1.014,0.977,1.027,8
3,py_func,361.277,369.849,0.994,0.963,1.073,8


trial: full / base


,shape,base ns,full ns,ratio median,ratio min,ratio max,rounds
0,trivial,153250.000,149874.500,1.050,0.879,1.077,8
1,unlifted_c,496.448,692.702,1.412,1.288,1.526,8
2,lifted_leaf,571.491,751.160,1.336,1.270,3.326,8
3,py_func,361.277,383.271,1.043,1.016,1.903,8


trial: gate / base


,shape,base ns,gate ns,ratio median,ratio min,ratio max,rounds
0,trivial,153250.000,149312.500,1.040,0.895,1.072,8
1,unlifted_c,496.448,531.247,1.085,1.012,1.146,8
2,lifted_leaf,571.491,611.823,1.091,1.034,1.125,8
3,py_func,361.277,366.398,1.049,0.979,1.077,8


## The noise floor against each candidate

For each set and shape: the A/A ratio's range, and each candidate's median ratio. A candidate's median inside the A/A
range is not distinguishable from no change on these rounds.

In [4]:
rows = []
for n in sets:
    aa = tables.get((n, "base_aa"))
    for s in SHAPES:
        row = {"set": n, "shape": s}
        if aa is not None:
            a = aa.set_index("shape").loc[s]
            row["A/A min"], row["A/A max"] = a["ratio min"], a["ratio max"]
        for c in ["full", "gate"]:
            if (n, c) in tables:
                m = tables[n, c].set_index("shape").loc[s, "ratio median"]
                row[f"{c} median"] = m
                if aa is not None:
                    row[f"{c} beyond noise"] = bool(m > row["A/A max"] or m < row["A/A min"])
        rows.append(row)
pd.DataFrame(rows).round(3)

,set,shape,A/A min,A/A max,full median,full beyond noise,gate median,gate beyond noise
0,release,trivial,0.972,1.051,1.047,False,1.048,False
1,release,unlifted_c,0.975,1.019,1.358,True,1.083,True
2,release,lifted_leaf,0.953,1.033,1.303,True,1.091,True
3,release,py_func,0.969,1.050,1.048,False,1.044,False
4,release-loaded,trivial,0.944,1.188,1.049,False,1.049,False
5,release-loaded,unlifted_c,0.857,1.205,1.476,True,1.106,False
6,release-loaded,lifted_leaf,0.966,1.215,1.415,True,1.131,False
7,release-loaded,py_func,0.871,1.588,1.082,False,1.058,False
8,trial,trivial,0.866,1.027,1.050,True,1.040,True
9,trial,unlifted_c,0.918,1.098,1.412,True,1.085,False


In [5]:
# ns per approved call that each candidate adds over base, under wasm32 (median of per-round differences)
diff = []
for n, wasm in sets.items():
    for c in [x for x in ["full", "gate", "base_aa"] if x in set(wasm.build)]:
        for s in ["unlifted_c", "lifted_leaf", "py_func"]:
            b = wasm[(wasm["shape"] == s) & (wasm.build == "base")].set_index("round").ns
            o = wasm[(wasm["shape"] == s) & (wasm.build == c)].set_index("round").ns
            diff.append({"set": n, "build": c, "shape": s, "added ns/call (median)": (o - b).median()})
pd.DataFrame(diff).pivot(index=["set", "shape"], columns="build", values="added ns/call (median)").round(1)

build                       base_aa   full  gate
set            shape                            
release        lifted_leaf     -2.9   73.1  21.8
               py_func          1.0    7.6   7.0
               unlifted_c      -1.2   74.1  17.4
release-loaded lifted_leaf     17.9  207.0  65.1
               py_func          1.5   26.4  18.4
               unlifted_c       9.3  209.4  47.3
trial          lifted_leaf      8.1  194.0  51.0
               py_func         -2.6   14.3  18.8
               unlifted_c     -18.9  200.7  42.0

## The trivial run's spread

A trivial run is about 150 µs under wasm32, mostly compile; its per-round ratio spreads wider than the call shapes', so
a median a few percent off 1 is not read as a cost.

In [6]:
spread = []
for n, wasm in sets.items():
    t = wasm[wasm["shape"] == "trivial"]
    for b, g in t.groupby("build"):
        spread.append({"set": n, "build": b, "min µs": g.ns.min() / 1e3, "median µs": g.ns.median() / 1e3,
                       "max µs": g.ns.max() / 1e3, "max/min": g.ns.max() / g.ns.min()})
pd.DataFrame(spread).round(2)

,set,build,min µs,median µs,max µs,max/min
0,release,base,54.92,57.94,58.79,1.07
1,release,base_aa,57.08,57.67,58.21,1.02
2,release,full,60.04,60.98,64.25,1.07
3,release,gate,60.08,60.69,61.08,1.02
4,release-loaded,base,129.25,129.56,159.17,1.23
5,release-loaded,base_aa,122.08,129.21,157.29,1.29
6,release-loaded,full,135.04,136.58,163.71,1.21
7,release-loaded,gate,118.12,136.44,161.79,1.37
8,trial,base,128.75,153.25,155.83,1.21
9,trial,base_aa,128.92,143.46,156.37,1.21


## The real cells: does a cell anyone wrote move?

Every real `python_exec` cell the corpus finds (`corpus.py`: the local run exports and the bench store; 1,702 that
parse), timed inside the dialect by `cells.py`, the way `python-dialect-run.ts` runs one, with `drive.mjs
--one=one-cells.mjs`: a fresh Pyodide per core per round, the order rotated, `base`/`base_aa` the same 7c40b038 core.
The cell text stays local (`export_cells.py` writes it outside the repo); the rounds hold only `corpus.py`'s
`cell_id` (a hash of the text), the outcome's kind and the ns.

A cell enters the ratio only when every process, on both cores, gave it the SAME outcome. The headline is the cells that
finish (`ok`); a cell that stops on its own error is timed too, but its run is cut short, so it gets its own row. A
cell's figure for one process is the min over its reps; its ratio is the median over rounds of the per-round ratio. The
decision (repl-specs, 2026-10-21): the full probe stands if the candidate's upper quartile is inside the A/A spread;
lane 13 opens if real cells move 1.2x or more.

In [7]:
cellsets = {}
for n in ("release", "release-loaded", "trial"):
    p = DATA / f"cells-rounds-{n}.json"
    if p.exists():
        cellsets[n] = pd.DataFrame([{"round": r["round"], "build": r["label"], "cell": cid, "outcome": v["outcome"],
                                     "ns": min(v["ns"])} for r in json.loads(p.read_text()) for cid, v in r["ns"].items()])

def per_cell(c, other):
    # one ratio per cell: the median over rounds of other/base, on cells whose outcome never varied across both builds
    two = c[c.build.isin(["base", other])]
    same = two.groupby("cell").outcome.nunique() == 1
    two = two[two.cell.isin(same[same].index)]
    w = two.pivot_table(index=["cell", "round"], columns="build", values="ns").dropna()
    r = (w[other] / w["base"]).groupby("cell").median()
    out = two.groupby("cell").outcome.first().to_frame().join(r.rename("ratio"), how="inner")
    return out.join(w["base"].groupby("cell").median().rename("base ns"))

rows = []
for n, c in cellsets.items():
    print(f"=== {n}: outcomes per process (round 0, base):", c[(c["round"] == 0) & (c.build == "base")].outcome.value_counts().to_dict())
    for o in ["base_aa", "full", "gate"]:
        if o not in set(c.build):
            continue
        pc = per_cell(c, o)
        for label, sel in [("ok", pc.outcome == "ok"), ("own error", pc.outcome.str.startswith("error:")),
                           ("ok, base >= 1 ms", (pc.outcome == "ok") & (pc["base ns"] >= 1e6))]:
            q = pc[sel].ratio
            rows.append({"set": n, "build": o, "cells": label, "n": len(q), "q25": q.quantile(.25), "median": q.median(),
                         "q75": q.quantile(.75), "q90": q.quantile(.9), "max": q.max()})
cells_tab = pd.DataFrame(rows).round(3)
cells_tab

=== release: outcomes per process (round 0, base): {'refused': 1225, 'error:KeyError': 233, 'ok': 217, 'error:TypeError': 18, 'error:IndexError': 8, 'error:ValueError': 1}


=== release-loaded: outcomes per process (round 0, base): {'refused': 1225, 'error:KeyError': 233, 'ok': 217, 'error:TypeError': 18, 'error:IndexError': 8, 'error:ValueError': 1}
=== trial: outcomes per process (round 0, base): {'refused': 1225, 'error:KeyError': 233, 'ok': 217, 'error:TypeError': 18, 'error:IndexError': 8, 'error:ValueError': 1}


,set,build,cells,n,q25,median,q75,q90,max
0,release,base_aa,ok,217,0.999,1.006,1.014,1.020,1.083
1,release,base_aa,own error,260,0.997,1.008,1.017,1.026,1.065
2,release,base_aa,"ok, base >= 1 ms",56,0.992,1.002,1.017,1.021,1.083
3,release,full,ok,217,1.026,1.036,1.045,1.060,1.224
4,release,full,own error,260,1.029,1.041,1.051,1.070,1.289
5,release,full,"ok, base >= 1 ms",56,1.020,1.028,1.045,1.060,1.224
6,release-loaded,base_aa,ok,217,0.973,0.991,1.003,1.025,1.122
7,release-loaded,base_aa,own error,260,0.969,0.991,1.007,1.035,1.177
8,release-loaded,base_aa,"ok, base >= 1 ms",78,0.967,0.987,1.004,1.032,1.122
9,release-loaded,full,ok,217,1.024,1.041,1.065,1.112,1.603


In [8]:
# The decision: each candidate's upper quartile against the A/A pair's interquartile spread, on the same cells.
verdict = []
for (n, label), g in cells_tab.groupby(["set", "cells"], sort=False):
    aa = g[g.build == "base_aa"]
    for _, r in g[g.build != "base_aa"].iterrows():
        lo, hi = (aa.q25.iloc[0], aa.q75.iloc[0]) if len(aa) else (float("nan"), float("nan"))
        verdict.append({"set": n, "cells": label, "build": r.build, "n": r.n, "median": r["median"], "q75": r.q75,
                        "A/A q25": lo, "A/A q75": hi, "q75 inside A/A": bool(lo <= r.q75 <= hi),
                        "moves >= 1.2x (q75)": bool(r.q75 >= 1.2)})
pd.DataFrame(verdict)

,set,cells,build,n,median,q75,A/A q25,A/A q75,q75 inside A/A,moves >= 1.2x (q75)
0,release,ok,full,217,1.036,1.045,0.999,1.014,False,False
1,release,own error,full,260,1.041,1.051,0.997,1.017,False,False
2,release,"ok, base >= 1 ms",full,56,1.028,1.045,0.992,1.017,False,False
3,release-loaded,ok,full,217,1.041,1.065,0.973,1.003,False,False
4,release-loaded,own error,full,260,1.045,1.073,0.969,1.007,False,False
5,release-loaded,"ok, base >= 1 ms",full,78,1.038,1.066,0.967,1.004,False,False
6,trial,ok,full,217,1.047,1.096,0.997,1.056,False,False
7,trial,own error,full,260,1.053,1.112,0.989,1.048,False,False
8,trial,"ok, base >= 1 ms",full,77,1.047,1.084,0.999,1.059,False,False
